# Auto-encoders with Keras

## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy
import scipy.interpolate
import tensorflow as tf
import tensorflow.keras as keras

## Data loading

We will use MNIST data normalized to values in $[0, 1]$ (instead of $[0, 255]$). Note that usually we prefer normalizing by centering and diving by the standard deviation, but this one makes visualization easier.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
nb_classes = 10
input_dim = 28 * 28
X_train = X_train.reshape(-1, input_dim).astype('float32')
X_test = X_test.reshape(-1, input_dim).astype('float32')

# 0 pixels stay at 0.
X_train = X_train / 255.0
X_test = X_test / 255.0

In [ ]:
X_train.shape

## Simple auto-encoders

### Auto-encoder creation

Let's design a simple auto-encoder…

In [ ]:
encoding_dim = 4
model = keras.models.Sequential()
model.add(keras.layers.InputLayer(input_shape=(input_dim,)))
model.add(keras.layers.Dense(encoding_dim, activation="leaky_relu"))
model.add(keras.layers.Dense(input_dim, activation="sigmoid"))
model.compile(optimizer="adam", loss="mean_squared_error")
model.summary()

… and train it

In [ ]:
model.fit(X_train, X_train,
          epochs=50,
          batch_size=256,
          validation_split=0.2)

Onto a more complex one now.

In [ ]:
encoder = keras.Sequential(
    [keras.layers.Input(shape=(input_dim,)),
     keras.layers.Dense(150, activation="leaky_relu"),
     keras.layers.Dense(150, activation="leaky_relu"),
     keras.layers.Dense(50,  activation="leaky_relu"),
     keras.layers.Dense(encoding_dim, activation="leaky_relu")],
    name="encoder")

decoder = keras.Sequential(
    [keras.layers.Input(shape=(encoding_dim,)),
     keras.layers.Dense(50,  activation="leaky_relu"),
     keras.layers.Dense(150, activation="leaky_relu"),
     keras.layers.Dense(150, activation="leaky_relu"),
     keras.layers.Dense(input_dim,
                        activation="sigmoid",
                        kernel_initializer="orthogonal")],
    name="decoder")

autoencoder = keras.Sequential(
    [keras.layers.Input(shape=(input_dim,)), encoder, decoder],
    name="autoencoder")
autoencoder.compile(optimizer="adam", loss="mean_squared_error")


encoder.summary()
decoder.summary()
autoencoder.summary()



In [ ]:
encoding_dim = 4

In [ ]:
autoencoder.fit(X_train, X_train,
          epochs=50,
          batch_size=256,
          validation_split=0.2)

### Prediction on white noise

In [ ]:
white_noise

In [ ]:
white_noise = numpy.random.random_sample((1, encoding_dim))
plt.imshow(decoder.predict(white_noise).reshape(28, 28), cmap="gray_r")
plt.show()

### Encoding test data

Let's encode unseen data into the code computed by the encoder

In [ ]:
codes = encoder.predict(X_test)

In [ ]:
codes.shape

### Computation of centroids in the code space for each digit

In [ ]:
means = numpy.vstack([codes[y_test == i].mean(axis=0)
                      for i in range(nb_classes)])
stds = numpy.vstack([codes[y_test == i].std(axis=0)
                     for i in range(nb_classes)])

for i in range(10):
  dimension_stats = [f"{mean:5.2f}±{std:.2f}"
                     for mean, std in zip(means[i], stds[i])]
  print(f"Digit {i} {', '.join(dimension_stats)}")

Decoding of the centroids by the decoder:

In [ ]:
f, ax = plt.subplots(1, nb_classes, figsize=(1.4 * nb_classes, 2))

centroid_images = decoder.predict(means).reshape(-1, 28, 28)

for i, centroid_image in enumerate(centroid_images):
  ax[i].imshow(centroid_image, cmap="gray_r")
  ax[i].axis("off")
plt.show()

### Latent walk between two digit centroids

Now that we know where are the centroids for each digit, we can go through the latent space in-between two of them.

In [ ]:
def latent_walk(start: int, end: int, n: int = 15):
  interpolator = scipy.interpolate.interp1d([0, n - 1],
                                            means[[start, end], :],
                                            axis=0)
  interpolated_codes = interpolator(range(n))
  interpolated_images = decoder.predict(interpolated_codes).reshape(-1, 28, 28)

  f, ax = plt.subplots(1, n, figsize=(n * 1.4, 2))
  for i, interpolated_image in enumerate(interpolated_images):
    ax[i].imshow(interpolated_image, cmap="gray_r")
    ax[i].axis("off")
  plt.show()


latent_walk(2, 6)
latent_walk(2, 7)
latent_walk(3, 5)

### Auto-encoding of the complete test base

In [ ]:
decoded_imgs = model.predict(X_test)

In [ ]:
decoded_imgs_train = model.predict(X_train)

### Visualization

In [ ]:
n = 15  # Number of digits to display

random_indexes = numpy.random.choice(decoded_imgs.shape[0],
                                     size=n,
                                     replace=False)

f, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for i, random_index in enumerate(random_indexes):
    # Original input is on the top row
    ax[0, i].imshow(X_test[random_index].reshape(28, 28), cmap="gray_r")
    ax[0, i].axis("off")

    # Reconstruction is on the bottom row
    ax[1, i].imshow(decoded_imgs[random_index].reshape(28, 28), cmap="gray_r")
    ax[1, i].axis("off")
plt.show()

In [ ]:
mse = keras.losses.MeanSquaredError(reduction="none")
test_anomalies = (-mse(decoded_imgs, X_test).numpy()).argsort()
train_anomalies = (-mse(decoded_imgs_train, X_train).numpy()).argsort()

n = 20

print("Pires reconstructions sur le train")
_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for i, index in enumerate(train_anomalies[:n]):
    # L'original en haut
    ax[0, i].set_title(y_train[index])
    ax[0, i].imshow(X_train[index].reshape(28, 28), cmap="gray_r")
    ax[0, i].axis("off")

    # La reconstruction en bas
    ax[1, i].imshow(decoded_imgs_train[index].reshape(28, 28), cmap="gray_r")
    ax[1, i].axis("off")
plt.show()


print("Pires reconstructions sur le test")
_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for i, index in enumerate(test_anomalies[:n]):
    # L'original en haut
    ax[0, i].set_title(y_test[index])
    ax[0, i].imshow(X_test[index].reshape(28, 28), cmap="gray_r")
    ax[0, i].axis("off")

    # La reconstruction en bas
    ax[1, i].imshow(decoded_imgs[index].reshape(28, 28), cmap="gray_r")
    ax[1, i].axis("off")
plt.show()

print("Meilleures reconstructions sur le train")
_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for i, index in enumerate(train_anomalies[-n:]):
    # L'original en haut
    ax[0, i].set_title(y_train[index])
    ax[0, i].imshow(X_train[index].reshape(28, 28), cmap="gray_r")
    ax[0, i].axis("off")

    # La reconstruction en bas
    ax[1, i].imshow(decoded_imgs_train[index].reshape(28, 28), cmap="gray_r")
    ax[1, i].axis("off")
plt.show()


print("Meilleures reconstructions sur le test")
_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for i, index in enumerate(test_anomalies[-n:]):
    # L'original en haut
    ax[0, i].set_title(y_test[index])
    ax[0, i].imshow(X_test[index].reshape(28, 28), cmap="gray_r")
    ax[0, i].axis("off")

    # La reconstruction en bas
    ax[1, i].imshow(decoded_imgs[index].reshape(28, 28), cmap="gray_r")
    ax[1, i].axis("off")
plt.show()

## Denoising auto-encoders

### Gaussian noise application

In [ ]:
noise_factor = 0.5
X_train_noisy = X_train + numpy.random.normal(0, noise_factor, X_train.shape)
X_test_noisy = X_test + numpy.random.normal(0, noise_factor, X_test.shape)

# Clipping to avoid values above 1 or below 0
numpy.clip(X_train_noisy, 0, 1, out=X_train_noisy)
numpy.clip(X_test_noisy, 0, 1, out=X_test_noisy)

In [ ]:
n = 10
f, ax = plt.subplots(1, n, figsize=(n * 1.4, 2))
for i in range(n):
    ax[i].imshow(X_test_noisy[i].reshape(28, 28), cmap="gray_r")
    ax[i].set_title(y_test[i])
    ax[i].axis("off")
plt.show()

### Denoising auto-encoder creation


In [ ]:
encoding_dim = 4
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=(input_dim,)))
model.add(keras.layers.Dense(encoding_dim, activation="relu"))
model.add(keras.layers.Dense(input_dim, activation="sigmoid"))
model.compile(optimizer="adam", loss="mean_squared_error")

### Training

In [ ]:
model.fit(X_train_noisy, X_train,
          epochs=50,
          batch_size=256,
          validation_split=0.2)

### Auto-encoding the complete test base

Autoencodez les images de test et stockez les images obtenues dans la variable `decoded_imgs`

In [ ]:
decoded_imgs = model.predict(X_test_noisy)

### Visualization

In [ ]:
n = 10
f, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for i in range(n):
    # L'original en haut
    ax[0, i].imshow(X_test_noisy[i].reshape(28, 28), cmap="gray_r")
    ax[0, i].set_title(str(y_test[i]))
    ax[0, i].axis("off")

    # La reconstruction en bas
    ax[1, i].imshow(decoded_imgs[i].reshape(28, 28), cmap="gray_r")
    ax[1, i].axis("off")
plt.show()

### Denoising convolutional auto-encoders

We first need to reshape to $(\text{batch} \times \text{width} \times \text{height} \times \text{channels})$ to use Keras.

In [ ]:
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)
X_train_noisy = X_train_noisy.reshape(-1, 28, 28, 1)
X_test_noisy = X_test_noisy.reshape(-1, 28, 28, 1)

### Denoising convolutional auto-encoder creation

In [ ]:
encoding_dim = 4

model = keras.models.Sequential()
# Encoding
model.add(keras.layers.InputLayer(input_shape=X_train.shape[1:]))
model.add(keras.layers.Conv2D(32, (3, 3), activation="leaky_relu", padding="same"))
model.add(keras.layers.Conv2D(32, (2, 2), (2, 2), activation="leaky_relu", padding="same"))
model.add(keras.layers.Conv2D(32, (3, 3), activation="leaky_relu", padding="same"))
model.add(keras.layers.Conv2D(32, (2, 2), (2, 2), activation="leaky_relu", padding="same"))
model.add(keras.layers.Conv2D(1, (3, 3), activation="leaky_relu", padding="same"))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(encoding_dim))

# Decoding
model.add(keras.layers.Dense(49))
model.add(keras.layers.Reshape((7, 7, 1)))
model.add(keras.layers.Conv2D(32, (3, 3), activation="leaky_relu", padding="same"))
model.add(keras.layers.Conv2DTranspose(32, (2, 2), (2, 2), activation="leaky_relu", padding="same"))
model.add(keras.layers.Conv2D(32, (3, 3), activation="leaky_relu", padding="same"))
model.add(keras.layers.Conv2DTranspose(32, (2, 2), (2, 2), activation="leaky_relu", padding="same"))
model.add(keras.layers.Conv2D(1, (3, 3), activation="sigmoid", padding="same"))

model.compile(optimizer=keras.optimizers.Adam(0.0001), loss="mean_squared_error")

model.summary()

### Training

In [ ]:
model.fit(X_train_noisy, X_train,
          epochs=100,
          batch_size=128,
          validation_split=0.2)

### Visualization

In [ ]:
decoded_imgs = model.predict(X_test_noisy).reshape(-1, 28, 28)

n = 10
f, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))

random_indexes = numpy.random.choice(X_test.shape[0],
                                     replace=False,
                                     size=n)

for i, random_index in enumerate(random_indexes):
    # Original on the top row
    ax[0, i].set_title(str(y_test[random_index]))
    ax[0, i].imshow(X_test_noisy[random_index].reshape(28, 28), cmap="gray_r")
    ax[0, i].axis("off")

    # Reconstruction on the bottom row
    ax[1, i].imshow(decoded_imgs[random_index], cmap="gray_r")
    ax[1, i].axis("off")
plt.show()

### Visualization on images on which we didn't add noise

In [ ]:
decoded_imgs = model.predict(X_test).reshape(-1, 28, 28)

n = 10
f, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))

random_indexes = numpy.random.choice(X_test.shape[0],
                                     replace=False,
                                     size=n)

for i, random_index in enumerate(random_indexes):
    # Original on the top row
    ax[0, i].set_title(str(y_test[random_index]))
    ax[0, i].imshow(X_test[random_index].reshape(28, 28), cmap="gray_r")
    ax[0, i].axis("off")

    # Reconstruction on the bottom row
    ax[1, i].imshow(decoded_imgs[random_index], cmap="gray_r")
    ax[1, i].axis("off")
plt.show()

## Variational auto-encoders

For VAEs, the training method is less straightforward (due to the need for the [reparametrization trick](https://stats.stackexchange.com/questions/199605/how-does-the-reparameterization-trick-for-vaes-work-and-why-is-it-important)). The suggestion by the Keras maintainers is to use the following code:

In [ ]:
class Sampling(keras.layers.Layer):
    """Uses (z_mean, z_log_var) to sample z, the vector encoding a digit."""

    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

In [ ]:
latent_dim = 2

encoder_inputs = keras.Input(shape=(28, 28, 1))
x = keras.layers.Conv2D(32, 3, activation="relu", strides=2, padding="same")(encoder_inputs)
x = keras.layers.Conv2D(64, 3, activation="relu", strides=2, padding="same")(x)
x = keras.layers.Flatten()(x)
x = keras.layers.Dense(16, activation="relu")(x)
z_mean = keras.layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = keras.layers.Dense(latent_dim, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])
encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")
encoder.summary()

In [ ]:
latent_inputs = keras.Input(shape=(latent_dim,))
x = keras.layers.Dense(7 * 7 * 64, activation="relu")(latent_inputs)
x = keras.layers.Reshape((7, 7, 64))(x)
x = keras.layers.Conv2DTranspose(64, 3, activation="relu", strides=2, padding="same")(x)
x = keras.layers.Conv2DTranspose(32, 3, activation="relu", strides=2, padding="same")(x)
decoder_outputs = keras.layers.Conv2DTranspose(1, 3, activation="sigmoid", padding="same")(x)
decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")
decoder.summary()

In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(
            name="reconstruction_loss"
        )
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    keras.losses.binary_crossentropy(data, reconstruction), axis=(1, 2)
                )
            )
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            total_loss = reconstruction_loss + kl_loss
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

In [ ]:
vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam())
vae.fit(X_train, epochs=30, batch_size=128)

In [ ]:
def auto_encode_example(example: numpy.ndarray) -> None:
  x_z_mean, x_z_log_var, x_z = vae.encoder(example[None, ...])
  decoded = vae.decoder(x_z).numpy().squeeze()
  plt.imshow(example.squeeze(), cmap="gray_r")
  plt.show()
  plt.imshow(decoded, cmap="gray_r")
  plt.show()


auto_encode_example(X_test[0])